In [1]:
import os
# Use the repository root as the working directory, whether this notebook is
# launched from the repo root or from the notebooks/ folder.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


# Notebook 07: SOC Workload Metrics

For the best model per dataset (highest macro-F1 in `results/master_results_all.csv`),
compared against the rule-based baseline (`results/baselines/baseline_metrics.csv`):

- False alerts per 1,000 alerts = (1 - precision) x 1,000
- False positives / false negatives normalised per 100,000 flows
- Missed-attack reduction relative to the rule baseline
- Analyst time saving = (baseline false positives - ML false positives) x 3 minutes

The time saving estimates analyst-minutes avoided by reducing false alerts relative
to the rule baseline. It is a relative reduction against the baseline, not absolute
analyst workload. Canonical output: `results/soc_workload.csv`

In [2]:
import numpy as np
import pandas as pd
print('Libraries loaded.')

Libraries loaded.


In [3]:
master = pd.read_csv('results/master_results_all.csv')
baselines = pd.read_csv('results/baselines/baseline_metrics.csv')
print(master.columns.tolist())
print(baselines)

['dataset', 'model', 'f1_macro', 'f1_attack', 'f1_benign', 'precision', 'recall', 'accuracy', 'auc_roc', 'pr_auc', 'mcc', 'balanced_acc', 'specificity', 'npv', 'fpr', 'tp', 'fp', 'fn', 'tn', 'training_time_sec', 'cv_f1_mean', 'cv_f1_std']
         dataset     model  f1_macro  precision    recall  accuracy       mcc  \
0  UGRansome2024  Baseline  0.627431   0.382812  0.705638  0.673548  0.314229   
1     CICIoT2023  Baseline  0.598641   0.996307  0.884248  0.883785  0.334668   

   balanced_acc  specificity       npv       fpr     tp    fp    fn    tn  
0      0.684855     0.664072  0.884261  0.335928   2891  4661  1206  9214  
1      0.874471     0.864693  0.153241  0.135307  34529   128  4520   818  


In [4]:
INVESTIGATION_TIME_PER_ALERT_MIN = 3  # standard SOC triage assumption

# Best model per dataset by macro-F1
best_idx = master.groupby('dataset')['f1_macro'].idxmax()
best = master.loc[best_idx]

rows = []
for _, brow in best.iterrows():
    dataset = brow['dataset']
    model   = brow['model']
    ml_tp = int(brow['tp']); ml_fp = int(brow['fp'])
    ml_fn = int(brow['fn']); ml_tn = int(brow['tn'])

    bl = baselines[baselines['dataset'] == dataset].iloc[0]
    bl_tp = int(bl['tp']); bl_fp = int(bl['fp'])
    bl_fn = int(bl['fn']); bl_tn = int(bl['tn'])

    n_test_flows   = ml_tp + ml_fp + ml_fn + ml_tn
    n_test_attacks = ml_tp + ml_fn

    ml_precision = ml_tp / (ml_tp + ml_fp) if (ml_tp + ml_fp) > 0 else 0.0
    bl_precision = bl_tp / (bl_tp + bl_fp) if (bl_tp + bl_fp) > 0 else 0.0

    ml_fa_per_1000 = (1 - ml_precision) * 1000
    bl_fa_per_1000 = (1 - bl_precision) * 1000
    fa_reduction_pct = ((bl_fa_per_1000 - ml_fa_per_1000) / bl_fa_per_1000 * 100
                        if bl_fa_per_1000 > 0 else 0.0)

    ml_fp_per_100k_flows = ml_fp / n_test_flows * 100_000
    bl_fp_per_100k_flows = bl_fp / n_test_flows * 100_000
    ml_fn_per_100k_attacks = ml_fn / n_test_attacks * 100_000 if n_test_attacks > 0 else 0.0
    bl_fn_per_100k_attacks = bl_fn / n_test_attacks * 100_000 if n_test_attacks > 0 else 0.0

    missed_attacks_reduction_pct = ((bl_fn - ml_fn) / bl_fn * 100) if bl_fn > 0 else 0.0

    # Corrected time saving: reduction in false alerts vs. baseline, not absolute ML FP.
    false_alerts_avoided = bl_fp - ml_fp
    time_saved_minutes = false_alerts_avoided * INVESTIGATION_TIME_PER_ALERT_MIN
    time_saved_hours = time_saved_minutes / 60

    rows.append({
        'dataset': dataset,
        'best_model': model,
        'baseline_fp': bl_fp,
        'baseline_fn': bl_fn,
        'ml_fp': ml_fp,
        'ml_fn': ml_fn,
        'n_test_flows': n_test_flows,
        'n_test_attacks': n_test_attacks,
        'baseline_false_alerts_per_1000': round(bl_fa_per_1000, 4),
        'ml_false_alerts_per_1000': round(ml_fa_per_1000, 4),
        'false_alerts_reduction_pct': round(fa_reduction_pct, 4),
        'baseline_fp_per_100k_flows': round(bl_fp_per_100k_flows, 4),
        'ml_fp_per_100k_flows': round(ml_fp_per_100k_flows, 4),
        'baseline_fn_per_100k_attack_flows': round(bl_fn_per_100k_attacks, 4),
        'ml_fn_per_100k_attack_flows': round(ml_fn_per_100k_attacks, 4),
        'missed_attacks_reduction_pct': round(missed_attacks_reduction_pct, 4),
        'false_alerts_avoided': false_alerts_avoided,
        'time_saved_minutes': time_saved_minutes,
        'time_saved_hours': round(time_saved_hours, 4),
    })

COLS = ['dataset','best_model','baseline_fp','baseline_fn','ml_fp','ml_fn',
        'n_test_flows','n_test_attacks',
        'baseline_false_alerts_per_1000','ml_false_alerts_per_1000','false_alerts_reduction_pct',
        'baseline_fp_per_100k_flows','ml_fp_per_100k_flows',
        'baseline_fn_per_100k_attack_flows','ml_fn_per_100k_attack_flows',
        'missed_attacks_reduction_pct','false_alerts_avoided',
        'time_saved_minutes','time_saved_hours']
df_soc = pd.DataFrame(rows)[COLS]
df_soc.to_csv('results/soc_workload.csv', index=False)
print('Saved results/soc_workload.csv (canonical)')
df_soc

Saved results/soc_workload.csv (canonical)


,dataset,best_model,baseline_fp,baseline_fn,ml_fp,ml_fn,n_test_flows,n_test_attacks,baseline_false_alerts_per_1000,ml_false_alerts_per_1000,false_alerts_reduction_pct,baseline_fp_per_100k_flows,ml_fp_per_100k_flows,baseline_fn_per_100k_attack_flows,ml_fn_per_100k_attack_flows,missed_attacks_reduction_pct,false_alerts_avoided,time_saved_minutes,time_saved_hours
0,CICIoT2023,RandomForest,128,4520,75,64,39995,39049,3.6933,1.9201,48.0112,320.0400,187.5234,11575.2004,163.8966,98.5841,53,159,2.65
1,UGRansome2024,XGBoost,4661,1206,64,7,17972,4097,617.1875,15.4068,97.5037,25934.7874,356.1095,29436.1728,170.8567,99.4196,4597,13791,229.85


In [5]:
print('=== SOC workload (best model per dataset) ===')
print(df_soc.to_string(index=False))
print()
print('Notebook 07 complete.')

=== SOC workload (best model per dataset) ===
      dataset   best_model  baseline_fp  baseline_fn  ml_fp  ml_fn  n_test_flows  n_test_attacks  baseline_false_alerts_per_1000  ml_false_alerts_per_1000  false_alerts_reduction_pct  baseline_fp_per_100k_flows  ml_fp_per_100k_flows  baseline_fn_per_100k_attack_flows  ml_fn_per_100k_attack_flows  missed_attacks_reduction_pct  false_alerts_avoided  time_saved_minutes  time_saved_hours
   CICIoT2023 RandomForest          128         4520     75     64         39995           39049                          3.6933                    1.9201                     48.0112                    320.0400              187.5234                         11575.2004                     163.8966                       98.5841                    53                 159              2.65
UGRansome2024      XGBoost         4661         1206     64      7         17972            4097                        617.1875                   15.4068                     97.50